# 02 – M0 (W2-05), M1 (W2-06), M1-h (W2-07)

Hướng dẫn từng bước: `docs/huong_dan/W2-05.md`, `W2-06.md`, `W2-07.md`.
Ô nào có chữ **TODO** là phần m tự viết. Nhớ: thử núm vặn thì chỉ nhìn điểm trên **dev**; tập thi (**test**) để dành tới bước cuối.

In [ ]:
# Ô 0 – chuẩn bị (lần nào mở notebook cũng chạy ô này trước; Colab hỏi quyền vào Drive thì bấm cho phép)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'      # lối tắt tới Drive chung (xem docs/huong_dan/README.md, mục đầu)
!git clone -q https://github.com/quanuiter/NLI.git /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt
import os
for sub in ['data/processed', 'tools', 'results']:
    target = os.path.join(DRIVE_DIR, sub); os.makedirs(target, exist_ok=True)
    if os.path.islink(sub): continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.symlink(target, sub)

In [ ]:
from collections import Counter
import re
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
from src.data import load_scenario
from src.embeddings import tokenize
from src.evaluate import evaluate_and_save, summarize
from src.utils import set_seed
set_seed(42)
data = {s: load_scenario(s) for s in ['S1', 'S2']}
{s: {k: (len(v) if not isinstance(v, dict) else {t: len(x) for t, x in v.items()}) for k, v in d.items()} for s, d in data.items()}

## M0 – luôn đoán nhãn nhiều nhất (W2-05): t viết sẵn, m bấm chạy

In [ ]:
for s, d in data.items():
    majority = Counter(d['train']['label']).most_common(1)[0][0]
    print(s, 'nhãn đa số của train:', majority)
    for test_name, df in d['test'].items():
        evaluate_and_save(df['label'], [majority] * len(df), model='M0', scenario=s, test_set=test_name,
                          seed=42, config={'majority_label': majority}, ids=df['id'])

## M1 – mô hình đếm từ: TF-IDF + Logistic Regression / Naive Bayes (W2-06)

Câu 1 và câu 2 được biến **riêng** thành dãy số TF-IDF rồi ghép lại, cộng thêm mấy **đặc điểm so sánh hai câu** do m viết.
Chỉ có TF-IDF thì mô hình không biết hai câu liên quan nhau ra sao: t thử rồi, trên ViANLI điểm còn thấp hơn đoán bừa (khoảng 0,2).

In [ ]:
NEG = re.compile(r'\b(không|chưa|chẳng|chả|đâu có)\b', re.IGNORECASE)

def compare_features(df):
    """Mỗi cặp câu -> 1 dòng số. Số nào cũng phải >= 0 (để dùng được cả Naive Bayes)."""
    rows = []
    for p, h in zip(df['premise'], df['hypothesis']):
        tp, th = set(t.lower() for t in tokenize(p)), set(t.lower() for t in tokenize(h))
        overlap = len(tp & th) / max(1, len(th))          # bao nhiêu % từ của câu 2 có trong câu 1
        # TODO a: đếm từ phủ định trong câu 1 và câu 2: len(NEG.findall(p)), len(NEG.findall(h))
        # TODO b: hai câu dài chênh nhau bao nhiêu âm tiết, chia cho 50:  abs(len(tokenize(p)) - len(tokenize(h))) / 50
        # TODO c: câu 2 có con số nào mà câu 1 không có? có -> 1.0, không -> 0.0  (dùng re.findall(r'\d+', ...))
        rows.append([overlap])                             # TODO: thêm a, b, c vào danh sách này
    return np.array(rows)

def make_vectorizers():
    return (TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
            TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True))

def features(df, vp, vh, fit=False):
    P = vp.fit_transform(df['premise']) if fit else vp.transform(df['premise'])
    H = vh.fit_transform(df['hypothesis']) if fit else vh.transform(df['hypothesis'])
    return sp.hstack([P, H, sp.csr_matrix(compare_features(df))]).tocsr()

In [ ]:
# Thử núm vặn C của Logistic Regression, nhìn điểm trên DEV. Xong S2 thì sửa s = 'S1' chạy lại
s = 'S2'
d = data[s]
vp, vh = make_vectorizers()
X_train = features(d['train'], vp, vh, fit=True)
X_dev = features(d['dev'], vp, vh)
for C in [0.1, 1, 10]:
    model = LogisticRegression(C=C, max_iter=2000).fit(X_train, d['train']['label'])
    print(f'C={C}  dev macro-F1 = {f1_score(d["dev"]["label"], model.predict(X_dev), average="macro"):.3f}')

**TODO d:** làm y vậy với Naive Bayes `MultinomialNB(alpha=...)`, thử alpha = 0.1, 0.5, 1.0. Ghi lại bảng điểm dev của cả 2 cách.

**TODO e:** với **mỗi** kịch bản, chọn cách + núm vặn có điểm dev cao nhất, rồi chấm trên tập thi bằng đoạn code này:
```python
for test_name, df in d['test'].items():
    evaluate_and_save(df['label'], best_model.predict(features(df, vp, vh)), model='M1', scenario=s,
                      test_set=test_name, seed=42, config={'clf': 'LR', 'C': best_C, 'features': '...'},
                      ids=df['id'], premise=df['premise'], hypothesis=df['hypothesis'])
```

In [ ]:
# TODO d, e

## M1-h – chỉ cho mô hình đọc câu 2 (W2-07)

**TODO f:** giống M1 nhưng **chỉ** dùng TF-IDF của câu 2 (không câu 1, không `compare_features`). Thử C trên dev,
lưu với `model='M1h'`. Nếu M1-h cao hơn hẳn M0 thì dữ liệu bị "lộ đề" qua câu 2. Code mở đầu có trong `docs/huong_dan/W2-07.md`.

In [ ]:
# TODO f

In [ ]:
summarize().query("model in ['M0', 'M1', 'M1h']")